In [1]:
# ============================================================
# Network Intrusion Detection System using ML  -  Colab demo
# Paste into ONE Colab cell and run (Runtime > Run all)
# ============================================================
!pip -q install plotly scikit-learn pandas numpy

import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from IPython.display import display, HTML
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---------- 1. Firewall log CSV (synthetic, same scale as the project) ----------
# To use your own CSV instead, set UPLOAD = True and upload it when prompted.
UPLOAD = False
rng = np.random.default_rng(42)
N, N_SCAN, N_SUS = 24753, 223, 1010          # 1,233 attacks = 4.98 %

countries = ["India","United States","China","Russia","Germany","Brazil","Netherlands","Iran","France","Vietnam"]
c_benign  = [.30,.25,.08,.03,.08,.06,.06,.02,.08,.04]
c_attack  = [.03,.14,.28,.24,.05,.05,.07,.09,.02,.03]
bad_ports = [4444,6667,31337,1337,23,445,3389,5900]
ok_ports  = [80,443,53,22,8080,25,110,993]

def ip(): return ".".join(map(str, rng.integers(1,255,4)))

def make(n, label):
    d = pd.DataFrame({"src_ip":[ip() for _ in range(n)], "dst_ip":[f"10.0.{rng.integers(0,6)}.{rng.integers(1,60)}" for _ in range(n)]})
    d["label"] = label
    if label == "Benign":
        d["country"]  = rng.choice(countries, n, p=c_benign)
        d["dst_port"] = rng.choice(ok_ports, n, p=[.25,.35,.15,.05,.08,.04,.04,.04])
        d["protocol"] = rng.choice(["TCP","UDP"], n, p=[.8,.2])
        d["action"]   = rng.choice(["allow","deny"], n, p=[.97,.03])
        d["bytes"]    = rng.lognormal(7.5,1.1,n).astype(int)
        d["packets"]  = rng.integers(2,60,n);   d["duration"] = rng.exponential(5,n)
        d["ports_touched"] = rng.integers(1,3,n)
    elif label == "Port Scan":
        d["country"]  = rng.choice(countries, n, p=c_attack)
        d["dst_port"] = rng.integers(1,10000,n)
        d["protocol"] = "TCP"; d["action"] = rng.choice(["deny","drop"], n, p=[.7,.3])
        d["bytes"]    = rng.integers(40,200,n); d["packets"] = rng.integers(1,4,n)
        d["duration"] = rng.exponential(.1,n);  d["ports_touched"] = rng.integers(15,200,n)
    else:
        d["country"]  = rng.choice(countries, n, p=c_attack)
        d["dst_port"] = rng.choice(bad_ports, n)
        d["protocol"] = rng.choice(["TCP","UDP"], n, p=[.85,.15])
        d["action"]   = rng.choice(["allow","deny"], n, p=[.45,.55])
        d["bytes"]    = rng.lognormal(10,1.2,n).astype(int)
        d["packets"]  = rng.integers(30,500,n); d["duration"] = rng.exponential(40,n)
        d["ports_touched"] = rng.integers(1,6,n)
    return d

logs = pd.concat([make(N-N_SCAN-N_SUS,"Benign"), make(N_SCAN,"Port Scan"), make(N_SUS,"Suspicious Connection")]).sample(frac=1, random_state=1).reset_index(drop=True)
logs["timestamp"] = pd.date_range("2026-06-01", periods=len(logs), freq="3min")

if UPLOAD:
    from google.colab import files
    up = files.upload(); logs = pd.read_csv(next(iter(up)))

# ---------- 2. Train ML model & classify ----------
feats = ["dst_port","bytes","packets","duration","ports_touched"]
X = pd.concat([logs[feats], pd.get_dummies(logs[["protocol","action"]])], axis=1)
Xtr, Xte, ytr, yte = train_test_split(X, logs["label"], test_size=.3, stratify=logs["label"], random_state=7)
model = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=7).fit(Xtr, ytr)
print("MODEL REPORT (30% hold-out)\n", classification_report(yte, model.predict(Xte), digits=3))

logs["prediction"] = model.predict(X)
logs["confidence"] = model.predict_proba(X).max(axis=1)
def severity(r):
    if r.prediction == "Benign": return "None"
    if r.prediction == "Port Scan": return "High" if r.ports_touched > 100 else "Medium"
    return "High" if r.dst_port in (4444,31337,1337) or r.bytes > 100000 else ("Medium" if r.bytes > 20000 else "Low")
logs["severity"] = logs.apply(severity, axis=1)
alerts = logs[logs.prediction != "Benign"]

# ---------- 3. KPI cards ----------
total, nal = len(logs), len(alerts); score = nal/total*100
risk = "Low" if score < 10 else "Medium" if score < 25 else "High"
rc = {"Low":"#2ecc71","Medium":"#f39c12","High":"#e74c3c"}[risk]
top = alerts.prediction.value_counts().idxmax()
cards = [("Total Logs",f"{total:,}","#4aa3ff"),("Benign",f"{total-nal:,}","#2ecc71"),("Alerts",f"{nal:,}","#e74c3c"),
         ("Threat Score",f"{score:.2f}%","#f39c12"),("Top Attack",top,"#b388ff"),("Risk Level",risk,rc)]
html = "<div style='background:#0e1117;padding:16px;border-radius:12px;font-family:sans-serif'>"
html += "<h2 style='color:#fff;margin:0 0 12px'>🛡️ Network Intrusion Detection System</h2><div style='display:flex;gap:10px;flex-wrap:wrap'>"
for t,v,c in cards:
    html += f"<div style='flex:1;min-width:140px;background:#1a1f2b;border-left:4px solid {c};padding:12px;border-radius:8px'><div style='color:#9aa4b5;font-size:12px'>{t}</div><div style='color:{c};font-size:22px;font-weight:700'>{v}</div></div>"
display(HTML(html + "</div></div>"))

# ---------- 4. Charts (Plotly, interactive) ----------
T = "plotly_dark"
d = alerts.prediction.value_counts().reset_index(); d.columns = ["Type","Count"]
f1 = px.pie(d, names="Type", values="Count", hole=.55, template=T, title="Attack Distribution")
f1.update_traces(textinfo="percent+label"); f1.show()

tr = logs.assign(Category=np.where(logs.prediction=="Benign","Benign",logs.prediction)).Category.value_counts().reset_index()
tr.columns = ["Category","Logs"]
px.bar(tr, x="Category", y="Logs", color="Category", template=T, title="Traffic Categories", text_auto=True).show()

fl = alerts.groupby(["src_ip","dst_ip"]).size().nlargest(10).reset_index(name="Hits")
fl["Flow"] = fl.src_ip + " → " + fl.dst_ip
px.bar(fl.sort_values("Hits"), x="Hits", y="Flow", orientation="h", template=T, title="Top 10 Attack Flows").show()

cc = alerts.country.value_counts().reset_index(); cc.columns = ["Country","Attacks"]
px.bar(cc, x="Country", y="Attacks", color="Attacks", template=T, title="Attacks by Country", color_continuous_scale="Reds").show()

px.scatter_geo(cc, locations="Country", locationmode="country names", size="Attacks", color="Attacks",
               projection="natural earth", template=T, color_continuous_scale="Reds",
               title="Attack Origin Map (Geo-IP)").show()

sv = alerts.severity.value_counts().reindex(["Low","Medium","High"]).fillna(0).reset_index(); sv.columns = ["Severity","Count"]
px.bar(sv, x="Severity", y="Count", color="Severity", template=T, title="Risk Score: Severity Counts",
       color_discrete_map={"Low":"#2ecc71","Medium":"#f39c12","High":"#e74c3c"}, text_auto=True).show()

# ---------- 5. Latest detection + recent history + report ----------
cols = ["timestamp","src_ip","dst_ip","protocol","dst_port","action","prediction","severity","confidence"]
print("\n🚨 LATEST DETECTION"); display(alerts.sort_values("timestamp").tail(1)[cols].style.hide(axis="index"))
print("📜 RECENT ATTACK HISTORY (last 15)"); display(alerts.sort_values("timestamp", ascending=False).head(15)[cols])

alerts[cols].to_csv("attack_report.csv", index=False)
print("\n✅ attack_report.csv saved")
try:
    from google.colab import files; files.download("attack_report.csv")
except Exception: pass

MODEL REPORT (30% hold-out)
                        precision    recall  f1-score   support

               Benign      0.999     1.000     0.999      7056
            Port Scan      1.000     1.000     1.000        67
Suspicious Connection      1.000     0.970     0.985       303

             accuracy                          0.999      7426
            macro avg      1.000     0.990     0.995      7426
         weighted avg      0.999     0.999     0.999      7426




🚨 LATEST DETECTION


timestamp,src_ip,dst_ip,protocol,dst_port,action,prediction,severity,confidence
2026-07-22 12:30:00,250.80.154.128,10.0.2.24,TCP,1337,allow,Suspicious Connection,High,1.000000


📜 RECENT ATTACK HISTORY (last 15)


,timestamp,src_ip,dst_ip,protocol,dst_port,action,prediction,severity,confidence
24730,2026-07-22 12:30:00,250.80.154.128,10.0.2.24,TCP,1337,allow,Suspicious Connection,High,1.000000
24727,2026-07-22 12:21:00,5.188.246.3,10.0.5.38,UDP,5900,deny,Suspicious Connection,Medium,1.000000
24690,2026-07-22 10:30:00,233.92.118.30,10.0.3.58,TCP,1337,deny,Suspicious Connection,High,1.000000
24672,2026-07-22 09:36:00,242.173.183.214,10.0.1.45,UDP,31337,deny,Suspicious Connection,High,1.000000
24666,2026-07-22 09:18:00,142.132.37.79,10.0.1.44,TCP,23,deny,Suspicious Connection,Low,1.000000
24660,2026-07-22 09:00:00,114.36.207.239,10.0.5.24,TCP,4444,deny,Suspicious Connection,High,0.993333
24635,2026-07-22 07:45:00,186.69.47.73,10.0.4.48,TCP,1337,deny,Suspicious Connection,High,0.953333
24612,2026-07-22 06:36:00,248.164.49.180,10.0.4.55,TCP,1337,deny,Suspicious Connection,High,1.000000
24582,2026-07-22 05:06:00,12.133.231.180,10.0.1.16,TCP,7661,deny,Port Scan,High,1.000000
24575,2026-07-22 04:45:00,58.230.201.169,10.0.1.21,TCP,445,allow,Suspicious Connection,Medium,0.993333



✅ attack_report.csv saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>